在工具内部访问运行时信息（如对话历史、用户数据和持久化记忆）

#### 工具内部访问 State（短期记忆）

访问对话历史，跟踪工具调用次数

In [8]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

@tool
def set_user_name(new_name: str, runtime: ToolRuntime) -> Command:
    """Set the user's name in the conversation state."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f'user name updated to: {new_name}',
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

class CustomState(AgentState):
    user_preferences: dict
    user_name: str

agent = create_agent(
    model='deepseek-chat',
    tools=[get_last_user_message, get_user_preference, set_user_name],
    state_schema=CustomState,
)

In [9]:
state = {
    'messages': [HumanMessage('What are my preferences.')],
    'user_preferences': {'style': 'technical', 'verbosity': 'detailed'},
}
state = agent.invoke(state)
state['messages'].append(HumanMessage('What was the last human message?'))
state = agent.invoke(state)
state['messages'].append(HumanMessage('Set the username to Tom'))
state = agent.invoke(state)

In [10]:
for msg in state['messages']:
    msg.pretty_print()

================================ Human Message =================================

What are my preferences.
================================== Ai Message ==================================

I can help you check your preferences, but I need to know which specific preference you'd like to see. Could you tell me what preference you're interested in? For example, you might want to check preferences like "theme", "language", "notifications", or other settings you've previously configured.

Once you let me know the specific preference name, I can look it up for you.
================================ Human Message =================================

What was the last human message?
================================== Ai Message ==================================

I can check what the last human message was for you.
Tool Calls:
  get_last_user_message (call_00_mdxFaXW70EoOy9eGrcoPTl2Q)
 Call ID: call_00_mdxFaXW70EoOy9eGrcoPTl2Q
  Args:
================================= Tool Message ===============

In [11]:
state.get('user_name')

'Tom'